# Unix-terminal. Настройка окружения и установка пакетов


## Мотивация

Python-проект — это не только код, но и версия интерпретатора, точные версии пакетов и их зависимостей. Пакет с тем же именем после обновления может вести себя иначе: например, код вокруг `timm` ожидает четыре промежуточные карты признаков модели, а получает три и ломается уже во время обучения. Отдельное окружение и зафиксированные зависимости позволяют воспроизвести рабочий запуск на другой машине, обновлять проект осознанно и быстро вернуться к исправной версии вместо долгого поиска причины «у меня работает».


## 1. Переменные оболочки, `export` и `source`

`NAME=value` задаёт переменную в текущей оболочке. `export NAME=value` передаёт её дочерним процессам. `NAME=value command` задаёт значение только для одной команды.

`source file` выполняет файл в текущей оболочке. Это используют для файлов окружения и активации virtual environment; перед `source` незнакомый файл следует прочитать.


In [ ]:
%%bash
mkdir -p /tmp/course-env
echo 'COURSE_NAME="linux and python"' > /tmp/course-env/course.env
echo 'export COURSE_DATA=/tmp/course-env/data' >> /tmp/course-env/course.env

source /tmp/course-env/course.env
echo "COURSE_NAME=$COURSE_NAME"
bash -c 'echo "COURSE_DATA=$COURSE_DATA"'


### Вопрос

Почему переменная из `course.env` остаётся после `source course.env`, но не после `bash course.env`?

<details>
<summary>Ответ</summary>

`source` выполняет присваивания в текущей оболочке. Отдельный Bash меняет только собственное окружение и после завершения исчезает.

</details>


## 2. `PATH`, `PYTHONPATH` и `LD_LIBRARY_PATH`

Пути поиска состоят из каталогов, разделённых двоеточиями. Порядок важен: каталоги проверяются слева направо.

- `PATH` — исполняемые команды;
- `PYTHONPATH` — Python-модули;
- `LD_LIBRARY_PATH` — динамические библиотеки Linux.

Новый каталог добавляют, сохраняя прежнее значение: `export PATH="$HOME/bin:$PATH"`. `which name` показывает найденную команду.

`python -c 'CODE'` выполняет переданную строку как Python-код. Это флаг Python, а не Bash.


In [ ]:
%%bash
rm -rf /tmp/course-path
mkdir -p /tmp/course-path/bin /tmp/course-path/python

echo '#!/usr/bin/env bash' > /tmp/course-path/bin/course-info
echo 'echo "course command: $COURSE_NAME"' >> /tmp/course-path/bin/course-info
chmod +x /tmp/course-path/bin/course-info
echo 'value = "module from course PYTHONPATH"' > /tmp/course-path/python/course_module.py

export COURSE_NAME=terminal
export PATH="/tmp/course-path/bin:$PATH"
export PYTHONPATH="/tmp/course-path/python:$PYTHONPATH"

which course-info
course-info
python3 -c 'import course_module; print(course_module.value)'


### Вопрос

Что произойдёт, если заменить `PATH="$HOME/bin:$PATH"` на `PATH="$HOME/bin"`?

<details>
<summary>Ответ</summary>

Стандартные каталоги исчезнут из поиска, и оболочка перестанет находить многие обычные команды.

</details>


## 3. Интерпретатор, `-m`, `venv` и `virtualenv`

`python -m module` просит выбранный интерпретатор найти модуль и выполнить его как программу. Поэтому `python -m pip` запускает pip именно выбранного Python.

`python3.12 -m venv .venv` создаёт каталог `.venv` в текущем каталоге. Внутри находятся отдельный Python, pip и каталог установленных пакетов. Имя `.venv` принято по соглашению, но может быть другим.

`source .venv/bin/activate` ставит `.venv/bin` в начало `PATH`; `deactivate` возвращает прежнее окружение.

`venv` входит в стандартную библиотеку, `virtualenv` устанавливается отдельно. Эти команды нужно узнавать в старых проектах; в практических задачах курса окружения создаём через `uv venv`.

```bash
python3 -m virtualenv --python=python3.12 .venv
```


In [ ]:
%%bash
rm -rf /tmp/course-venv
python3 -m venv /tmp/course-venv

echo 'before:'
which python3
python3 --version
source /tmp/course-venv/bin/activate
echo 'inside:'
which python
python --version
python -m pip --version
deactivate


### Вопрос

Какой Python создаст окружение после `python3.12 -m venv .venv` и что означает `-m`?

<details>
<summary>Ответ</summary>

Окружение будет основано на запущенном Python 3.12. `-m` находит указанный модуль в этом интерпретаторе и выполняет его как программу.

</details>


## 4. Зависимости Python


### Прямые, транзитивные и зафиксированные зависимости

Прямая зависимость указана проектом. Транзитивная нужна другой зависимости. Ограничение версии задаёт допустимый диапазон, lock-файл фиксирует полный разрешённый набор версий.


### `uv` — основной рабочий вариант

`uv` управляет интерпретаторами, окружениями и зависимостями проекта. `uv venv --python 3.12 .venv` создаёт окружение на Python 3.12 и при необходимости загружает интерпретатор. `uv python pin 3.12` записывает версию проекта в `.python-version`.

`pyproject.toml` хранит прямые зависимости и ограничения, `uv.lock` — полный разрешённый набор версий. `.venv` не переносят между машинами: её восстанавливают командой `uv sync`.

`uv run command` запускает команду в окружении проекта без ручной активации. `uv tree` показывает дерево зависимостей.

```bash
# Создать проект и выбрать Python 3.12.
uv init
uv python pin 3.12

# Добавить requests и зафиксировать зависимости.
uv add requests
uv lock

# Восстановить .venv и запустить Python внутри неё.
uv sync
uv run python -c 'import requests; print(requests.__version__)'
uv tree
```


### `pip` — совместимость со старыми проектами

`pip` по-прежнему встречается в существующих проектах и устанавливает пакеты в выбранный интерпретатор. `pip freeze` печатает установленные версии, `pip install -r file` восстанавливает список из `requirements.txt`.

Для новых задач этого курса используем `uv`; команды pip нужны, чтобы понимать уже существующие проекты.

```bash
python -m pip install -r requirements.txt
```


### `conda` — специализированный вариант

`conda` полезна, когда проект зависит от conda-каналов или сложных нативных библиотек. `-n course` задаёт имя окружения, `python=3.12` — версию Python.

```bash
conda create -n course python=3.12 requests
conda activate course
conda env export > environment.yml
```

Для обычного Python-проекта в этом курсе используем `uv`.


In [ ]:
%%bash
rm -rf /tmp/course-uv
mkdir /tmp/course-uv
cd /tmp/course-uv

# Создать минимальный проект.
uv init

# Добавить зависимость; uv обновит pyproject.toml и uv.lock.
uv add requests

# Восстановить окружение по lock-файлу.
uv sync

# Запустить Python в окружении проекта без source activate.
uv run python -c 'import requests; print(requests.__version__)'


### Вопрос

Какие файлы uv нужно передать на другую машину и какой командой восстановить окружение? Нужно ли копировать `.venv`?

<details>
<summary>Ответ</summary>

Передают `pyproject.toml` и `uv.lock`, затем выполняют `uv sync`. Каталог `.venv` не копируют: он зависит от путей и платформы и пересоздаётся из описания проекта.

</details>


## 5. Области установки и системные менеджеры

- Проект: `.venv`, созданная uv. Зависимости изолированы от других проектов.
- Пользователь: Homebrew и пользовательские установки утилит не привязаны к одному проекту.
- Система: `apt`/`apt-get` в Debian и Ubuntu, `dnf` в Fedora/RHEL, `pacman` в Arch.
- Изолированные приложения: snap устанавливает приложение вместе с его окружением.

`apt` удобен для интерактивной работы, `apt-get` имеет более стабильный интерфейс для скриптов. `apt update` обновляет сведения о доступных версиях, `apt upgrade` обновляет установленные пакеты. `install`, `remove`, `purge` устанавливают, удаляют и удаляют вместе с конфигурацией.

`apt list --installed` показывает установленные пакеты. `apt list --upgradable` показывает доступные обновления; список актуален только после свежего `apt update`.


In [ ]:
%%bash
apt list --installed \
  > installed-packages.txt 2> apt-errors.txt

apt list --upgradable \
  > upgradable-packages.txt 2>> apt-errors.txt


### Вопрос

Почему `apt list --upgradable` может показать устаревший результат и какая команда обновляет эти сведения?

<details>
<summary>Ответ</summary>

Команда читает локальный индекс репозиториев. Его обновляет `apt update`; без свежего индекса список доступных версий может устареть.

</details>


## 6. Динамические библиотеки: `ldd` и `ldconfig`

Исполняемый файл может загружать общие библиотеки во время запуска. `ldd program` показывает необходимые библиотеки и найденные файлы. В примере ниже проверяется обычная системная программа `/usr/bin/env`.

`ldconfig -p` выводит кэш библиотек загрузчика. `LD_LIBRARY_PATH` временно добавляет каталог поиска для запускаемой программы; глобальную конфигурацию через него обычно не строят.

`not found` в выводе `ldd` означает, что зависимость известна, но подходящий файл не найден.


In [ ]:
%%bash
ldd /usr/bin/env 2>/dev/null | head -n 12
ldconfig -p 2>/dev/null | head -n 5
echo "LD_LIBRARY_PATH=${LD_LIBRARY_PATH:-not set}"


### Вопрос

Что означает `libexample.so => not found` в выводе `ldd`?

<details>
<summary>Ответ</summary>

Программа требует библиотеку, но динамический загрузчик не нашёл подходящий файл в известных путях.

</details>


## 7. Службы и журналы `systemd`

`systemctl` управляет системными службами, `journalctl` читает их журналы.

- `status` — состояние службы;
- `is-system-running` — общее состояние systemd;
- `start`, `stop`, `restart` — управление сейчас;
- `enable` — запуск при старте системы;
- `list-units --type=service` — загруженные службы;
- `list-unit-files --type=service` — установленные описания служб;
- `--no-pager` — вывести результат прямо в терминал;
- `journalctl -u NAME` — журнал службы;
- `-n 50` — последние 50 строк, `-f` — новые строки в реальном времени.


In [ ]:
%%bash
# Версия systemd.
systemctl --version | head -n 2

# Несколько доступных системных служб.
systemctl list-unit-files --type=service --no-pager 2>/dev/null | head -n 8

# Последние сообщения одной службы.
journalctl -u systemd-journald -n 5 --no-pager 2>/dev/null || true


### Вопрос

Служба завершилась с ошибкой. Где посмотреть её состояние и последние 50 строк журнала?

<details>
<summary>Ответ</summary>

Состояние: `systemctl status NAME`. Журнал: `journalctl -u NAME -n 50`.

</details>


## Дополнительно


### Что такое unit

Unit-файл — конфигурация для systemd. В service-unit записано, какую команду systemd должен запустить и как следить за её выполнением.

```ini
[Unit]
Description=Show service user

[Service]
Type=oneshot
ExecStart=/usr/bin/id
```

При `systemctl start course-report.service` systemd один раз запускает `/usr/bin/id`. Команда печатает пользователя и его группы в журнал, после чего завершается. Фоновая служба не остаётся работать.

- `Description` — подпись, которую покажет `systemctl status`;
- `Type=oneshot` — завершение команды ожидаемо: unit выполняет одно действие;
- `ExecStart=/usr/bin/id` — команда запуска. Указывается абсолютный путь, чтобы systemd не искал программу через пользовательский `PATH`.

`systemctl status course-report.service` показывает результат запуска, `journalctl -u course-report.service` — вывод команды. `systemctl cat course-report.service` печатает конфигурацию и дополнительные настройки, которые systemd нашёл для этого unit.


### Версии Python в uv

Эти команды решают разные задачи:

- `uv python install 3.12` — установить управляемый Python;
- `uv python pin 3.12` — записать выбор проекта в `.python-version`;
- `uv python find 3.12` — показать путь к подходящему интерпретатору.

`pin` не активирует окружение и не устанавливает зависимости.

```bash
uv python install 3.12
uv python pin 3.12
uv python find 3.12
```


### Обновление одной зависимости

`uv tree` показывает прямые и транзитивные зависимости. `uv lock --upgrade-package requests` пересчитывает `uv.lock`, разрешая обновить `requests`; ограничения из `pyproject.toml` продолжают действовать. `uv sync` применяет новый lock-файл к `.venv`.

```bash
uv tree
uv lock --upgrade-package requests
uv sync
```


### Подключение APT-репозитория

APT получает пакеты из настроенных источников. Стороннему источнику нужны адрес и ключ проверки подписей. На примере Docker для Ubuntu:

1. Установить `ca-certificates` и `curl`.
2. Сохранить ключ в `/etc/apt/keyrings/docker.asc`.
3. Создать `/etc/apt/sources.list.d/docker.sources`.
4. Выполнить `apt update` и установить пакеты.

В `.sources`: `URIs` — адрес, `Suites` — версия Ubuntu, `Components` — ветка репозитория, `Architectures` — архитектура, `Signed-By` — ключ проверки подписи.

`sudo` выполняет команду с правами администратора. `install -d` создаёт каталог, `-m 0755` задаёт права. `curl -fsSL` останавливается при HTTP-ошибке, скрывает прогресс и следует перенаправлениям. `tee` записывает root-файл, потому что обычное `>` выполняла бы текущая оболочка. `dpkg --print-architecture` печатает архитектуру пакетов текущей системы.


In [ ]:
%%bash
# Установить инструменты и создать каталог ключей.
sudo apt update
sudo apt install ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings

# Скачать ключ репозитория.
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg \
  -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Создать описание источника с реальными значениями Ubuntu и архитектуры.
sudo tee /etc/apt/sources.list.d/docker.sources <<EOF
Types: deb
URIs: https://download.docker.com/linux/ubuntu
Suites: $(source /etc/os-release && echo "${UBUNTU_CODENAME:-$VERSION_CODENAME}")
Components: stable
Architectures: $(dpkg --print-architecture)
Signed-By: /etc/apt/keyrings/docker.asc
EOF

# Обновить индекс и установить Docker.
sudo apt update
sudo apt install docker-ce docker-ce-cli containerd.io \
  docker-buildx-plugin docker-compose-plugin


### Диагностика окружения

Отчёт собирают по слоям, чтобы было видно источник проблемы:

- shell: `PATH`, `PYTHONPATH`, `LD_LIBRARY_PATH`;
- Python: `uv --version`, `uv python find`;
- система: `apt list`, `ldconfig -p`, `systemctl is-system-running`.

stdout каждого слоя сохраняют в отдельный файл, общий stderr — в `errors.txt`. Тогда пустой или ошибочный раздел не смешивается с остальными.
